# Coral Reef and Tourism 2

## Data Loading Note

The datasets used in this project are not included in this repository because redistribution permissions have not been fully verified.

To rerun this notebook, place the required dataset files inside the `data/` folder of the project.

Expected files:

- `data/MetricsVisits_new.csv`
- `data/MetricsVisits_new_expected.csv`

The notebook uses relative file paths from the `notebooks/` folder to the `data/` folder.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path("../data")

main_data_path = DATA_DIR / "MetricsVisits_new.csv"
expected_data_path = DATA_DIR / "MetricsVisits_new_expected.csv"

if not main_data_path.exists():
    raise FileNotFoundError(
        f"Required dataset not found: {main_data_path}\n"
        "Datasets are not included in this repository because redistribution permissions "
        "have not been verified. To rerun this notebook, place the required CSV files "
        "inside the data/ folder."
    )

if not expected_data_path.exists():
    raise FileNotFoundError(
        f"Required dataset not found: {expected_data_path}\n"
        "Datasets are not included in this repository because redistribution permissions "
        "have not been verified. To rerun this notebook, place the required CSV files "
        "inside the data/ folder."
    )

data = pd.read_csv(main_data_path)
data_expected = pd.read_csv(expected_data_path)

data.head()

In [ ]:
data.head()

In [ ]:
# Convert to PeriodIndex with quarterly frequency
data["Year-Quarter"] = data["Year-Quarter"].str.replace(" ", "-")
data["Year-Quarter"] = pd.PeriodIndex(data["Year-Quarter"], freq="Q")

# Convert to timestamp (default is end of quarter)
data["YQ"] = data["Year-Quarter"].dt.to_timestamp()

In [ ]:
data=data.drop(['Quarter','Year','Year-Quarter'],axis=1)
data.info()

In [ ]:
# Define categorical columns that represent discrete values
categorical = ['Quarters']

# Define continuous (numerical) columns that represent measurable quantities
continuous = ['AvgTurbidity','AvgCyclone_Frequency','AvgClimSST','AvgSSTA','AvgSTA_Frequency','AvgSSTA_DHW','AvgTSA','AvgTSA_Frequency',
             'AvgTSA_DHW','GSP','GDP','overnight_trips']

# Convert categorical columns to the "category" data type for efficient processing

for i in categorical:
    data[i] = data[i].astype("category")

data.info()

### Correlation Matrix

In [ ]:
# Compute the correlation matrix for continuous numerical variables
corr=data[continuous].corr()
# Create a figure and axis for the heatmap with a specified size
f,ax=plt.subplots(figsize=(12,8))
# Generate a heatmap to visualize correlations between numerical features
# 'annot=True' displays correlation values inside the heatmap cells
sns.heatmap(corr,cmap="crest",annot=True)

In [ ]:
sns.boxplot(data=data['overnight_trips'])
plt.suptitle('overnight_trips Boxplot')

### Define the test and train set

In [ ]:
from sklearn.model_selection import train_test_split

label = 'overnight_trips'
features = [column for column in data.columns if column != label and column != 'YQ']

X, y = data[features], data[label]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=40) # Hold out validation on the outside

X_train.shape,y_train.shape

### Average
Calculate the MAE of a simple model (like always predicting the average) and see if complex model beats it.

In [ ]:
from sklearn.metrics import mean_absolute_error

mae_avg = mean_absolute_error(y_test, [y_train.mean()]*len(y_test))
print(mae_avg)

## Linear Regression with Forward Stepwise Feature Selection

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold

categorical = ['Quarters']
continuous = ['AvgTurbidity','AvgCyclone_Frequency','AvgClimSST','AvgSSTA','AvgSTA_Frequency','AvgSSTA_DHW','AvgTSA','AvgTSA_Frequency',
             'AvgTSA_DHW','GSP','GDP','overnight_trips']

#support for 1-hot-encoded features using get_model
#We separate the categorical from the numerical
# def get_model(features):
#     categorical_features = [col for col in features if col in categorical]
#     numerical_features = [col for col in features if col in continuous]
#     return make_pipeline(
#         ColumnTransformer(transformers=[
#             ('categorical', OneHotEncoder(), categorical_features),
#             ('numerical', StandardScaler(), numerical_features)
#         ]),
#         LinearRegression())

def get_model(selected_features):
    categorical_selected = [f for f in selected_features if f in categorical]
    numerical_selected = [f for f in selected_features if f in continuous]

    transformers = []

    if len(categorical_selected) > 0:
        transformers.append(
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_selected)
        )

    if len(numerical_selected) > 0:
        transformers.append(
            ("numerical", StandardScaler(), numerical_selected)
        )

    preprocessor = ColumnTransformer(transformers=transformers)

    model = make_pipeline(
        preprocessor,
        LinearRegression()
    )

    return model

# 5-fold CV
kfold = KFold(n_splits=5, shuffle=True, random_state=0)

### Variable selection with stepwise forward selection

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error
import numpy as np

class Solution:
    features = list()
    mae = float('Inf')

    def __init__(self, y):
        pred = [y.mean()] * len(y)
        self.mae = mean_absolute_error(y, pred)

    def update(self, features, mae):
        if mae < self.mae:
            self.features = features
            self.mae = mae
            return True
        return False


all_features = list(features)
current_features = list() #start with empty model
best = Solution(y) #initialize solution

while current_features != all_features:
    selected_feature = None

    for feature in set(all_features) - set(current_features):
        new_features = current_features + [feature] #add one feature at a time
        maes = cross_val_score(
            estimator=get_model(new_features),
            X=X_train[new_features], y=y_train,
            cv=kfold, scoring='neg_mean_absolute_error')
        mae = -np.average(maes)

        if best.update(new_features, mae):
            selected_feature = feature

    if selected_feature:
        current_features.append(selected_feature)
    else:
        break

best_features_lin = best.features

print('Selected features: ', end='')
print(', '.join(best_features_lin))
print(f'MAE feature selection: {round(mae,4)}')


In [ ]:
categorical_features_lin = []
numerical_features_lin = []

for feature in best_features_lin:
    for cat_feature in categorical:
        if feature == cat_feature:
            categorical_features_lin.append(feature)
    for num_feature in continuous:
        if feature == num_feature:
            numerical_features_lin.append(feature)

linear_reg = make_pipeline(
    ColumnTransformer(transformers=[
        ('categorical', OneHotEncoder(), categorical_features_lin),
        ('numerical', StandardScaler(), numerical_features_lin)
    ]),
    LinearRegression())

# Train model with selected features
linear_reg.fit(X_train[best_features_lin], y_train)

# Make predictions
y_pred_lin = linear_reg.predict(X_test)

# Calculate MAE
print(f"MAE linear regression: {mean_absolute_error(y_test, y_pred_lin)}")

In [ ]:
for element in list(zip(linear_reg.steps[1][1].coef_, best_features_lin)):
    print(f"Coeff {element[1]}: {element[0]}")

## Lasso degree 2

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Lasso

kfold = KFold(n_splits=5, shuffle=True, random_state=0)

categorical = ['Quarters']
continuous = ['AvgTurbidity','AvgCyclone_Frequency','AvgClimSST','AvgSSTA','AvgSTA_Frequency','AvgSSTA_DHW','AvgTSA','AvgTSA_Frequency',
             'AvgTSA_DHW','GSP','GDP']

Lasso_reg_d2= make_pipeline(
    ColumnTransformer(transformers=[
        ('categorical', OneHotEncoder(), categorical),
        ('numerical', StandardScaler(), continuous)
    ]),
    PolynomialFeatures(degree=2),
    GridSearchCV(estimator=Lasso(max_iter=100000),
                 param_grid=dict(alpha=np.logspace(-3, 5, 50)),
                 cv=kfold, scoring='neg_mean_absolute_error')
)

# Train model with selected features
Lasso_reg_d2.fit(X_train, y_train)

# Make predictions
y_pred_lasso_d2 = Lasso_reg_d2.predict(X_test)

# Calculate MAE
mean_absolute_error(y_test, y_pred_lasso_d2)

In [ ]:
y_pred_lasso_d2

In [ ]:
# Get the best Lasso model from GridSearchCV
best_lasso_d2 = Lasso_reg_d2.named_steps['gridsearchcv'].best_estimator_

# Get coefficients
coefficients_lasso_d2 = best_lasso_d2.coef_

print(f"Best alpha: {Lasso_reg_d2.named_steps['gridsearchcv'].best_params_['alpha']}")
print(f"\nTotal features: {len(coefficients_lasso_d2)}")
print(f"Non-zero coefficients: {np.sum(coefficients_lasso_d2 != 0)}")

In [ ]:
column_transform = Lasso_reg_d2.named_steps['columntransformer']
poly = Lasso_reg_d2.named_steps['polynomialfeatures']

# Get feature names from ColumnTransformer
feature_names = column_transform.get_feature_names_out()

# Get polynomial feature names
poly_feature_names = poly.get_feature_names_out(feature_names)

# Create DataFrame with non-zero coefficients
coef_df_lasso_d2 = pd.DataFrame({'feature': poly_feature_names, 'coefficient': coefficients_lasso_d2})
print("Non-zero coefficients:")
display(coef_df_lasso_d2[coef_df_lasso_d2["coefficient"]!=0])

## Lasso degree 1

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=0)

categorical = ['Quarters']
continuous = ['AvgTurbidity','AvgCyclone_Frequency','AvgClimSST','AvgSSTA','AvgSTA_Frequency','AvgSSTA_DHW','AvgTSA','AvgTSA_Frequency',
             'AvgTSA_DHW','GSP','GDP']

Lasso_reg_d1= make_pipeline(
    ColumnTransformer(transformers=[
        ('categorical', OneHotEncoder(), categorical),
        ('numerical', StandardScaler(), continuous)
    ]),
    GridSearchCV(estimator=Lasso(max_iter=100000),
                 param_grid=dict(alpha=np.logspace(-3, 5, 50)),
                 cv=kfold, scoring='neg_mean_absolute_error')
)

# Train model with selected features
Lasso_reg_d1.fit(X_train, y_train)

# Make predictions
y_pred_lasso_d1 = Lasso_reg_d1.predict(X_test)

# Calculate MAE
mean_absolute_error(y_test, y_pred_lasso_d1)

In [ ]:
# Get the best Lasso model from GridSearchCV
best_lasso_d1 = Lasso_reg_d1.named_steps['gridsearchcv'].best_estimator_

# Get coefficients
coefficients_lasso_d1 = best_lasso_d1.coef_

print(f"Best alpha: {Lasso_reg_d1.named_steps['gridsearchcv'].best_params_['alpha']}")
print(f"\nTotal features: {len(coefficients_lasso_d1)}")
print(f"Non-zero coefficients: {np.sum(coefficients_lasso_d1 != 0)}")

# Get feature names
column_transform = Lasso_reg_d1.named_steps['columntransformer']
feature_names = column_transform.get_feature_names_out()

# Create DataFrame with non-zero coefficients
coef_df_lasso_d1 = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients_lasso_d1})
display(coef_df_lasso_d1[coef_df_lasso_d1["coefficient"]!=0])

## Ridge

In [ ]:
from sklearn.linear_model import Ridge

kfold = KFold(n_splits=5, shuffle=True, random_state=0)

categorical = ['Quarters']
continuous = ['AvgTurbidity','AvgCyclone_Frequency','AvgClimSST','AvgSSTA','AvgSTA_Frequency','AvgSSTA_DHW','AvgTSA','AvgTSA_Frequency',
             'AvgTSA_DHW','GSP','GDP']

Ridge_reg= make_pipeline(
    ColumnTransformer(transformers=[
        ('categorical', OneHotEncoder(), categorical),
        ('numerical', StandardScaler(), continuous)
    ]),
    GridSearchCV(estimator=Ridge(max_iter=100000),
                 param_grid=dict(alpha=np.logspace(-3, 5, 50)),
                 cv=kfold, scoring='neg_mean_absolute_error')
)

# Train model with selected features
Ridge_reg.fit(X_train, y_train)

# Make predictions
y_pred_ridge = Ridge_reg.predict(X_test)

# Calculate MAE
mean_absolute_error(y_test, y_pred_ridge)

In [ ]:
# Get the best Ridge model from GridSearchCV
best_ridge = Ridge_reg.named_steps['gridsearchcv'].best_estimator_

# Get coefficients
coefficients = best_ridge.coef_

print(f"Best alpha: {Ridge_reg.named_steps['gridsearchcv'].best_params_['alpha']}")
print(f"\nTotal features: {len(coefficients)}")
print(f"Non-zero coefficients: {np.sum(coefficients != 0)}")

# Get feature names
column_transform = Ridge_reg.named_steps['columntransformer']
feature_names = column_transform.get_feature_names_out()

# Create DataFrame with non-zero coefficients
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})
display(coef_df[coef_df["coefficient"]!=0])

## Predict on new data set

A new data set is provided for the year 2018, with expected metrics (provided by NOAA) and overnight trips (forecasted by Australian Trade and Investment Commission). The idea is to compare that forecast with the prediction of this model.

In [ ]:
data_expected.head()

In [ ]:
label_exp = 'overnight_trips'

features_exp = ['Quarters', 'AvgSSTA', 'AvgCyclone_Frequency', 'AvgTSA_Frequency', 'AvgSSTA_DHW', 'AvgTurbidity',
                'AvgClimSST', 'AvgTSA_DHW', 'AvgSTA_Frequency','AvgTSA','GSP','GDP']

X_exp, y_exp = data_expected[features_exp], data_expected[label_exp]

X_exp.shape,y_exp.shape

In [ ]:
Last_year=['2017-01-01 00:00:00','2017-04-01 00:00:00','2017-07-01 00:00:00','2017-10-01 00:00:00']
# Last_year_df = data[data["YQ"].isin(Last_year)]
data["YQ"] = pd.to_datetime(data["YQ"])
Last_year_df = data[data["YQ"].dt.year == 2017]
print(f"Overnight trips in 2017: {Last_year_df['overnight_trips'].sum()}")
print(f"Expected overnight trips in 2018: {data_expected["overnight_trips"].sum()}")
# print(f"Difference in percentage: {round(((data_expected["overnight_trips"].sum() - Last_year_df['overnight_trips'].sum())
#                            /Last_year_df['overnight_trips'].sum()) * 100,2)}%")
expected_total = data_expected["overnight_trips"].sum()
last_year_total = Last_year_df["overnight_trips"].sum()

difference_percentage = ((expected_total - last_year_total) / last_year_total) * 100

print(f"Difference in percentage: {round(difference_percentage, 2)}%")

#### Linear Regression

In [ ]:
# Make predictions
y_pred_lin_exp = linear_reg.predict(X_exp)

In [ ]:
# Show results
results_linear = pd.DataFrame({'y_exp': y_exp, 'model prediction': y_pred_lin_exp})
totals = results_linear.sum()
pct_diff = ((totals['model prediction'] - totals['y_exp'])/ totals['y_exp']) * 100
results_totals_linear = results_linear.copy()
results_totals_linear.loc['Total'] = totals
#results_totals_linear.loc['% Diff'] = [None, pct_diff]

Diff_2018_exp = round(((y_exp.sum() - Last_year_df['overnight_trips'].sum())
                       /Last_year_df['overnight_trips'].sum()) * 100,2)
Diff_2018_pred = round(((y_pred_lin_exp.sum() - Last_year_df['overnight_trips'].sum())
                       /Last_year_df['overnight_trips'].sum()) * 100,2)

results_totals_linear.loc['% Diff with 2017'] = [Diff_2018_exp, Diff_2018_pred]

display(results_totals_linear)